# Exploratory Data Analysis

En este notebook comenzamos el proyecto con la exploración de las series de tiempo suministradas, las cuales indican los precios históricos de los materiales X, Y y Z.

El dataset proporcionado para el material X es el que más datos contiene, abarcando desde 1986 hasta 2024. Los materiales Y y Z, por su parte, tienen datos desde 2010 hasta 2023.

Para el análisis se evaluarán únicamente los datos que coinciden en una franja temporal específica entre 2010 y 2023 para todos los materiales. Esto permitirá calcular los valores históricos de los equipos 1 y 2 en este rango de tiempo y, posteriormente, realizar el forecasting para estos equipos en los años siguientes. Se verificó que los precios de los materiales estén muestreados en las mismas fechas, encontrándose 3530 coincidencias de fechas entre los datos.

Aunque el plan inicial de aproximación a la tarea consideraba hacer forecasting de los precios de las materias primas de forma independiente, se observa que su comportamiento presenta picos atípicos no periódicos y no linealidades significativas, lo cual representa un reto importante para dichas técnicas. Este enfoque generaría una incertidumbre compuesta más alta para la estimación de los precios de los equipos. Por tal motivo, se plantea la realización del forecasting directamente sobre los precios históricos de los equipos, considerando que se dispone de suficientes datos históricos de estos.

In [2]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

In [3]:
# Load X.csv (comma separator, standard format)
df_x = pd.read_csv("../data/raw/X.csv")
print("X.csv - Primeras filas:")
print(df_x.head())
print(f"\nShape: {df_x.shape}")
print(f"Data types:\n{df_x.dtypes}\n")
print("-" * 50)

# Load Y.csv (semicolon separator, European decimal format with comma)
df_y = pd.read_csv("../data/raw/Y.csv", sep=";")
print("\nY.csv - Primeras filas:")
print(df_y.head())
print(f"\nShape: {df_y.shape}")
print(f"Data types:\n{df_y.dtypes}\n")
print("-" * 50)

# Load Z.csv (comma separator, inverted column order)
df_z = pd.read_csv("../data/raw/Z.csv")
print("\nZ.csv - Primeras filas:")
print(df_z.head())
print(f"\nShape: {df_z.shape}")
print(f"Data types:\n{df_z.dtypes}\n")

X.csv - Primeras filas:
         Date  Price
0  2024-04-04  89.18
1  2024-04-03  89.35
2  2024-04-02  88.92
3  2024-04-01  87.42
4  2024-03-28  87.48

Shape: (9144, 2)
Data types:
Date      object
Price    float64
dtype: object

--------------------------------------------------

Y.csv - Primeras filas:
        Date   Price
0  12/9/2023  547,33
1  11/9/2023     546
2   8/9/2023     545
3   7/9/2023     550
4   6/9/2023   552,5

Shape: (4485, 2)
Data types:
Date     object
Price    object
dtype: object

--------------------------------------------------

Z.csv - Primeras filas:
     Price        Date
0  2225.25  2010-01-01
1  2225.25  2010-01-04
2  2246.50  2010-01-05
3  2302.50  2010-01-06
4  2306.50  2010-01-07

Shape: (3565, 2)
Data types:
Price    float64
Date      object
dtype: object



Se observa que el formato de los archivos .csv suministrados es diferente en todos los casos, presentando variaciones en el formato de las fechas, el separador decimal y la posición de las columnas. Por esta razón, se realizó un preprocesamiento estandarizado para homogeneizar los datos, guardando finalmente estos archivos procesados para su correcto manejo en análisis posteriores.

In [4]:
os.makedirs("../data/processed", exist_ok=True)

# Process Material X
df_x_clean = df_x.copy()
df_x_clean["Date"] = pd.to_datetime(df_x_clean["Date"], format="%Y-%m-%d")
df_x_clean["Price"] = df_x_clean["Price"].astype(float)
df_x_clean = df_x_clean.sort_values("Date").reset_index(drop=True)
df_x_clean = df_x_clean.drop_duplicates(subset=["Date"], keep="first")
df_x_clean[["Date", "Price"]].to_csv("../data/processed/X.csv", index=False)
print(
    f"X processed: {len(df_x_clean)} records from {df_x_clean['Date'].min()} to {df_x_clean['Date'].max()}"
)

# Process Material Y
df_y_clean = df_y.copy()
df_y_clean["Price"] = (
    df_y_clean["Price"].astype(str).str.replace(",", ".").astype(float)
)
df_y_clean["Date"] = pd.to_datetime(df_y_clean["Date"], format="%d/%m/%Y")
df_y_clean = df_y_clean.sort_values("Date").reset_index(drop=True)
df_y_clean = df_y_clean.drop_duplicates(subset=["Date"], keep="first")
df_y_clean[["Date", "Price"]].to_csv("../data/processed/Y.csv", index=False)
print(
    f"Y processed: {len(df_y_clean)} records from {df_y_clean['Date'].min()} to {df_y_clean['Date'].max()}"
)

# Process Material Z
df_z_clean = df_z.copy()
df_z_clean["Date"] = pd.to_datetime(df_z_clean["Date"], format="%Y-%m-%d")
df_z_clean["Price"] = df_z_clean["Price"].astype(float)
df_z_clean = df_z_clean.sort_values("Date").reset_index(drop=True)
df_z_clean = df_z_clean.drop_duplicates(subset=["Date"], keep="first")
df_z_clean[["Date", "Price"]].to_csv("../data/processed/Z.csv", index=False)
print(
    f"Z processed: {len(df_z_clean)} records from {df_z_clean['Date'].min()} to {df_z_clean['Date'].max()}"
)

X processed: 9144 records from 1988-06-27 00:00:00 to 2024-04-04 00:00:00
Y processed: 4485 records from 2006-07-11 00:00:00 to 2023-09-12 00:00:00
Z processed: 3565 records from 2010-01-01 00:00:00 to 2023-08-31 00:00:00


En el resultado de la celda anterior, pueden observarse la cantidad de registros de cada dataset y los intervalos de fecha en los que se encuentran estos mismos para cada uno de los materiales.

A continuación se revisarán algunas estadísticas de los precios de cada uno de estos materiales, así como la verificación de valores nulos dentro de los registros, para obtener una comprensión más robusta de cómo está conformado cada dataset y cuánta variación existe entre los datos suministrados en cada uno de ellos.

In [5]:
print("DESCRIPTIVE STATISTICS")

for df, name in [(df_x_clean, "X"), (df_y_clean, "Y"), (df_z_clean, "Z")]:
    print(f"\n{name} - Price Statistics:")
    print(df["Price"].describe())
    print(f"Missing values: {df['Price'].isna().sum()}")
    print("-" * 60)

DESCRIPTIVE STATISTICS

X - Price Statistics:
count    9144.000000
mean       51.323922
std        32.994964
min         9.640000
25%        19.687500
50%        45.940000
75%        75.207500
max       146.080000
Name: Price, dtype: float64
Missing values: 0
------------------------------------------------------------

Y - Price Statistics:
count    4485.000000
mean      565.458283
std       145.150530
min       257.500000
25%       485.000000
50%       543.670000
75%       617.670000
max      1062.370000
Name: Price, dtype: float64
Missing values: 0
------------------------------------------------------------

Z - Price Statistics:
count    3565.000000
mean     2037.081197
std       372.976076
min      1421.500000
25%      1767.500000
50%      1974.750000
75%      2235.750000
max      3984.000000
Name: Price, dtype: float64
Missing values: 0
------------------------------------------------------------


In [11]:
fig = make_subplots(
    rows=3,
    cols=1,
    subplot_titles=("Material X", "Material Y", "Material Z"),
    vertical_spacing=0.1,
)

# Material X
fig.add_trace(
    go.Scatter(
        x=df_x_clean["Date"],
        y=df_x_clean["Price"],
        mode="lines",
        name="Material X",
        line=dict(width=2),
    ),
    row=1,
    col=1,
)

# Material Y
fig.add_trace(
    go.Scatter(
        x=df_y_clean["Date"],
        y=df_y_clean["Price"],
        mode="lines",
        name="Material Y",
        line=dict(width=2),
    ),
    row=2,
    col=1,
)

# Material Z
fig.add_trace(
    go.Scatter(
        x=df_z_clean["Date"],
        y=df_z_clean["Price"],
        mode="lines",
        name="Material Z",
        line=dict(width=2),
    ),
    row=3,
    col=1,
)

# Update layout
fig.update_xaxes(title_text="Fecha", row=3, col=1)
fig.update_yaxes(title_text="Costo", row=1, col=1)
fig.update_yaxes(title_text="Costo", row=2, col=1)
fig.update_yaxes(title_text="Costo", row=3, col=1)

fig.update_layout(
    height=600,
    title_text="Precio de los materiales - Data Histórica",
    showlegend=True,
    template="plotly_white",
)

fig.show()

Se comprueba que las series de tiempo están muestreadas en el marco de días y presentan intervalos de muestreo diferentes en cada caso. Además, se observa un comportamiento con picos atípicos y zonas de alta no linealidad, lo que incrementa significativamente la complejidad de la tarea de forecasting para la estimación de costos del proyecto. Por tal motivo, y tras corroborar que existe un número suficiente de fechas coincidentes entre los tres materiales, se decide realizar el forecasting directamente sobre los precios históricos de los equipos en lugar de pronosticar cada materia prima de forma independiente y estimar el costo de los equipos con base en las predicciones, dado que esto entregaría una incertidumbre compuesta potencialmente más alta. Este enfoque reduce la propagación de error y aprovecha mejor los patrones históricos de los equipos.

In [8]:
# Find overlapping date range
start_date = max(
    df_x_clean["Date"].min(), df_y_clean["Date"].min(), df_z_clean["Date"].min()
)
end_date = min(
    df_x_clean["Date"].max(), df_y_clean["Date"].max(), df_z_clean["Date"].max()
)

print(f"Common date range: {start_date} to {end_date}")

# Filter each dataframe to common date range
df_x_filtered = df_x_clean[
    (df_x_clean["Date"] >= start_date) & (df_x_clean["Date"] <= end_date)
].copy()
df_y_filtered = df_y_clean[
    (df_y_clean["Date"] >= start_date) & (df_y_clean["Date"] <= end_date)
].copy()
df_z_filtered = df_z_clean[
    (df_z_clean["Date"] >= start_date) & (df_z_clean["Date"] <= end_date)
].copy()

# Merge dataframes on Date to align prices
df_merged = df_x_filtered[["Date", "Price"]].rename(columns={"Price": "Price_X"})
df_merged = df_merged.merge(
    df_y_filtered[["Date", "Price"]].rename(columns={"Price": "Price_Y"}),
    on="Date",
    how="inner",
)
df_merged = df_merged.merge(
    df_z_filtered[["Date", "Price"]].rename(columns={"Price": "Price_Z"}),
    on="Date",
    how="inner",
)

# Calculate Equipment costs
# Equipment 1: 20% material X + 80% material Y
df_merged["Equipment_1"] = 0.20 * df_merged["Price_X"] + 0.80 * df_merged["Price_Y"]

# Equipment 2: 33.33% each material (X, Y, Z)
df_merged["Equipment_2"] = (
    df_merged["Price_X"] + df_merged["Price_Y"] + df_merged["Price_Z"]
) / 3

print(f"\nEquipment costs calculated for {len(df_merged)} common dates")
print(
    f"\nEquipment 1 - Price range: ${df_merged['Equipment_1'].min():.2f} to ${df_merged['Equipment_1'].max():.2f}"
)
print(f"Equipment 1 - Average price: ${df_merged['Equipment_1'].mean():.2f}")
print(
    f"\nEquipment 2 - Price range: ${df_merged['Equipment_2'].min():.2f} to ${df_merged['Equipment_2'].max():.2f}"
)
print(f"Equipment 2 - Average price: ${df_merged['Equipment_2'].mean():.2f}")

Common date range: 2010-01-01 00:00:00 to 2023-08-31 00:00:00

Equipment costs calculated for 3530 common dates

Equipment 1 - Price range: $213.22 to $863.76
Equipment 1 - Average price: $460.04

Equipment 2 - Price range: $577.03 to $1668.05
Equipment 2 - Average price: $890.35


In [9]:
fig = go.Figure()

# Equipment 1
fig.add_trace(
    go.Scatter(
        x=df_merged["Date"],
        y=df_merged["Equipment_1"],
        mode="lines",
        name="Equipo 1",
        line=dict(width=3),
    )
)

# Equipment 2
fig.add_trace(
    go.Scatter(
        x=df_merged["Date"],
        y=df_merged["Equipment_2"],
        mode="lines",
        name="Equipo 2",
        line=dict(width=3),
    )
)

fig.update_layout(
    title="Valor de los equipos a lo largo del tiempo",
    xaxis_title="Fecha",
    yaxis_title="Costo",
    template="plotly_white",
    height=500,
    hovermode="x unified",
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
)

fig.show()

In [14]:
df_merged.to_csv("../data/processed/equipment_costs.csv", index=False)
print("Equipment costs saved to '../data/processed/equipment_costs.csv'")
print(
    f"\nDataset contains {len(df_merged)} records from {df_merged['Date'].min()} to {df_merged['Date'].max()}"
)

Equipment costs saved to '../data/processed/equipment_costs.csv'

Dataset contains 3530 records from 2010-01-04 00:00:00 to 2023-08-31 00:00:00
